"""
Q1 — Batch LLM calls: bounded concurrency, ordered results, retries with backoff.

Covers the base question AND every interviewer follow-up:
  1. Max N requests in flight            -> asyncio.Semaphore
  2. Results in input order              -> asyncio.gather preserves order
  3. Exponential backoff + jitter        -> _backoff_delay()
  4. Honour the API's Retry-After header -> RateLimitError.retry_after
  5. Retry only retryable errors         -> RETRYABLE tuple (429 / 5xx / timeout), never 400
  6. Hanging requests                    -> per-call timeout via asyncio.wait_for
  7. One failure must not kill the batch -> return_exceptions=True
  8. "Now it's 100,000 prompts"          -> batch_llm_queue(): fixed worker pool + asyncio.Queue

Run:  python q1_batch_llm.py
"""

In [ ]:
from __future__ import annotations

import asyncio
import time
import random
from dataclasses import dataclass

random.random()

0.22130058827592525

# Errors the LLM client can raise

In [ ]:
class RateLimitError(Exception):
  """ HTTP 429. The API may tell us how long to wait (Retry After header)."""

  def __init__(self,msg:str="rate limited",retry_after:float | None =None):
    super().__init__(msg)
    self.retry_after = retry_after

class ServerError(Exception):
    """HTTP 5xx — transient, worth retrying."""


class BadRequestError(Exception):
    """HTTP 400 — our fault. Retrying will never fix it."""


RETRYABLE  = (RateLimitError,ServerError,asyncio.TimeoutError)


@dataclass(frozen=True)
class RetryConfig :
  max_concurrency : int =5
  max_attempt :int = 5
  base_delay : float = 1.0
  max_delay : float = 30.0
  timeout : float  = 30.0


def _backoff_delay(attempt:int,cfg : RetryConfig,err : Exception) -> float :
  """Retry-After wins if the server sent one; otherwise exponential + jitter."""

  if isinstance(err, RateLimitError) and err.retry_after is not None:
     return err.retry_after

  exp  = cfg.base_delay*2**(attempt -1)
  jitter = random.uniform(0,exp)
  return  min(exp+jitter,cfg.max_delay)


async def _call_with_retry(prompt : str, sem : asyncio.Semaphore,
                           cfg : RetryConfig ) -> str :


          for attempt in range(1, cfg.max_attempt +1):
              async with sem :
                try :
                  return await asyncio.wait_for(call_llm(prompt),timeout = cfg.timeout)

                except BadRequestError:
                  raise

                except RETRYABLE as err :
                   if attempt  == cfg.max_attempt :
                      raise


                   delay = _backoff_delay(attempt,cfg,err)


              await asyncio.sleep(delay)

          raise RuntimeError("Unreachable")




async def batch_llm(prompts: list[str], cfg: RetryConfig = RetryConfig()) -> list[str | Exception]:

       sem = asyncio.Semaphore(cfg.max_concurrency)
       tasks = [_call_with_retry(p,sem,cfg) for p in prompts]



       return await asyncio.gather(*tasks , return_exceptions = True)

In [ ]:
import os

# Correctly retrieve the environment variable from os.environ
api_key = os.environ.get("GROQ_API_KEY")
print("GROQ_API_KEY in environment:", api_key is not None)

GROQ_API_KEY in environment: True


In [ ]:
import os
from groq import AsyncGroq

# Get the API key from environment variables or assign the string directly
api_key = "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"

# Set the environment variable so other components can access it if needed
os.environ["GROQ_API_KEY"] = api_key

client = None
if api_key:
    client = AsyncGroq(api_key=api_key)
else:
    print("Warning: GROQ_API_KEY environment variable is not set. Please set it before calling call_llm().")

async def call_llm(prompt: str) -> str:
    global client
    if client is None:
        # Check again in case it was set after cell execution
        current_key = os.environ.get("GROQ_API_KEY")
        if not current_key:
            raise ValueError("GROQ_API_KEY is missing. Please set the environment variable GROQ_API_KEY.")
        client = AsyncGroq(api_key=current_key)

    msg = await client.chat.completions.create(
        model="openai/gpt-oss-120b",
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.choices[0].message.content

In [ ]:
# ---------------- 50 prompts ----------------
prompts = [
    "What is Python?",
    "Explain list vs tuple in Python.",
    "What is a Python decorator?",
    "Explain generators and yield.",
    "What is the GIL in Python?",
    "Difference between async and threading?",
    "What is a large language model?",
    "Explain the transformer architecture simply.",
    "What is self-attention?",
    "What are embeddings?",
    "What is tokenization in LLMs?",
    "What is a context window?",
    "Explain temperature in LLM sampling.",
    "What is top-p sampling?",
    "What is prompt engineering?",
    "What is few-shot prompting?",
    "What is chain-of-thought prompting?",
    "What is RAG (retrieval augmented generation)?",
    "How does a vector database work?",
    "What is cosine similarity?",
    "How do you chunk documents for RAG?",
    "What is hybrid search?",
    "What is re-ranking in RAG?",
    "How do you evaluate a RAG system?",
    "What is hallucination in LLMs?",
    "How can you reduce hallucinations?",
    "What is fine-tuning?",
    "Fine-tuning vs RAG: when to use which?",
    "What is LoRA?",
    "What is quantization?",
    "What is an AI agent?",
    "What is the ReAct pattern?",
    "What is tool calling / function calling?",
    "How does an agent loop work?",
    "What is agent memory?",
    "What is a multi-agent system?",
    "What is MCP (Model Context Protocol)?",
    "How do you stop an agent from looping forever?",
    "What are guardrails in LLM apps?",
    "What is prompt injection?",
    "How do you stream LLM responses?",
    "How do you handle API rate limits?",
    "What is exponential backoff?",
    "How do you cache LLM responses?",
    "How do you reduce LLM cost?",
    "How do you reduce LLM latency?",
    "What is structured output / JSON mode?",
    "How do you validate LLM output with Pydantic?",
    "What is LLM observability?",
    "How do you deploy an LLM app to production?",
]


# ---------------- Run ----------------
async def main():
    print(f"Sending {len(prompts)} prompts (max 5 at a time)...\n")
    results = await batch_llm(prompts)
    print(results)
    print("\n---------------- Results ----------------")
    for i, (p, r) in enumerate(zip(prompts, results), 1):
        status = "FAILED" if isinstance(r, Exception) else "OK"
        print(f"{i:2}. [{status}] {r if status == 'OK' else type(r).__name__}")


# In Jupyter / Google Colab, we can directly await async functions
await main()

Sending 50 prompts (max 5 at a time)...

['**Python** is a high‑level, interpreted programming language known for its readability and flexibility. Created by\u202fGuido van\u202fRossum and first released in 1991, Python emphasizes clear, human‑readable syntax (e.g., using indentation to define code blocks) which makes it easy to learn and maintain.\n\n### Key characteristics\n\n| Feature | Why it matters |\n|---------|----------------|\n| **Simple, expressive syntax** | Reduces boilerplate; code often reads like plain English. |\n| **Interpreted** | No separate compile step; you can run scripts directly. |\n| **Dynamic typing** | Variable types are inferred at runtime, allowing rapid prototyping. |\n| **Rich standard library** | Built‑in modules for file I/O,', '**List vs. Tuple in Python – a quick side‑by‑side guide**\n\n| Feature | **list** | **tuple** |\n|---------|----------|-----------|\n| **Syntax** | `my_list = [1, 2, 3]` | `my_tuple = (1, 2, 3)` (or `my_tuple = 1, 2, 3`) |\n| *